In [12]:
import random
import pandas as pd

# Two assumptions
- any damage string is called an occurrence, whether it's successful or not
- to separate occurrences, we add the letter A to the string to indicate which ones are modified by a trigger check

In [13]:
def is_attack(occ):
    if isinstance(occ, str) and occ.upper().endswith('A'):
        return int(occ[:-1]), True
    return int(occ), False

In [14]:
def simulate_process(main_deck_size, climax_number, occurrences_parsed, trigger_soul, trigger_blank):
    # Main deck definition, with its discard pile
    deck = ['Climax'] * main_deck_size + ['Non-Climax'] * (main_deck_size - climax_number)
    random.shuffle(deck)
    discard = []
 
    # Trigger deck definition, with its discard pile
    trigger_deck = ['Soul trigger'] * trigger_soul + ['No soul trigger'] * trigger_blank
    random.shuffle(trigger_deck)
    trigger_discard = []
 
    # Clock zone and total damage count initialisation
    clock_zone = []
    total_kept = 0
 
    for base_k, is_A in occurrences_parsed:
        k = base_k
 
        # Trigger system
        if is_A:
            if not trigger_deck and trigger_discard:
                trigger_deck = trigger_discard[:]
                trigger_discard = []
                random.shuffle(trigger_deck)
            if trigger_deck:
                card = trigger_deck.pop()
                trigger_discard.append(card)
                if card == 'Soul trigger':
                    k += 1
 
        # Standard occurrence system
        drawn_this_occurrence = []
        occurrence_failed = False
        halted = False
        draws_done = 0
 
        while draws_done < k:
            if len(deck) == 0:
                # Refresh penalty system
                if not discard:
                    halted = True
                    break
                new_deck = discard[:]
                discard = []
                random.shuffle(new_deck)
                clock_card = new_deck.pop() 
                clock_zone.append(clock_card)
                total_kept += 1 
                deck = new_deck
                if len(deck) == 1:
                    halted = True
                    break
                continue
 
            card = deck.pop()
            drawn_this_occurrence.append(card)
            draws_done += 1
            if card == 'Climax':
                occurrence_failed = True
                break
 
        if halted:
            discard.extend(drawn_this_occurrence)
            break
 
        if occurrence_failed:
            discard.extend(drawn_this_occurrence)
        else:
            # In case no climax is revealed, all revealed cards go to clock zone and the total count goes up accordingly
            total_kept += len(drawn_this_occurrence)
            clock_zone.extend(drawn_this_occurrence)
 
            # Level-up system
            while len(clock_zone) >= 7:
                batch = clock_zone[:7]
                clock_zone = clock_zone[7:]
                white_idx = next((i for i, c in enumerate(batch) if c == 'Non-Climax'), None)
                # We assume here the player always put a non-climax out of the game
                if white_idx is not None:
                    batch.pop(white_idx)
                else:
                    batch.pop(0)
                discard.extend(batch)
 
    return total_kept

In [15]:
def simulation(deck_size_list, climax_list, occurrences, trigger_soul, trigger_blank, n_trials=100000):
    """
    Lance une simulation de type MonteCarlo avec les paramètres indiqués et renvoie un DataFrame correspondant
 
    Parameters
    ----------
    deck_size_list : list[int]        main deck size list
    climax_list : list[int]        climax number list
    occurrences : list        damage string to test
    trigger_soul : int        cards with soul triggers in trigger deck
    trigger_blank : int       cards without soul triggers in trigger deck
    n_trials : int            Monte Carlo simulations for each [N,p] combination
 
    Result
    ------
    Each size correspond to a [main_deck_size,climax_number] combination
    Each column correspond to the associated probability where at least X damage went through during simulation
    """
    occurrences_parsed = [is_attack(o) for o in occurrences]
    max_possible = sum(k + (1 if is_A else 0) for k, is_A in occurrences_parsed)
 
    rows = []
    for main_deck_size in deck_size_list:
        for climax_number in climax_list:
            totals = [
                simulate_process(main_deck_size, climax_number, occurrences_parsed, trigger_soul, trigger_blank)
                for _ in range(n_trials)
            ]
            row = {'Main deck size': main_deck_size, 'Climax number': climax_number}
            for x in range(0, max_possible + 1):
                row[f'X_min={x}'] = round(sum(1 for t in totals if t >= x) / n_trials, 4)
            rows.append(row)
 
    return pd.DataFrame(rows)

In [17]:
# usage exemple, feel free to use it as you wish
table = simulation(
        deck_size_list=[20,25,30],
        climax_list=[4,6,8],
        occurrences=[4,'3A',4,'3A',4,'3A'],
        trigger_soul = 15,
        trigger_blank = 35,
        n_trials=10000,
        )

table.to_csv('resultats_simulation.csv', index=False)
table

,Main deck size,Climax number,X_min=0,X_min=1,X_min=2,X_min=3,X_min=4,X_min=5,X_min=6,X_min=7,...,X_min=15,X_min=16,X_min=17,X_min=18,X_min=19,X_min=20,X_min=21,X_min=22,X_min=23,X_min=24
0,20,4,1.0,0.2635,0.2635,0.2635,0.1244,0.0196,0.0196,0.0133,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,20,6,1.0,0.2095,0.2095,0.2095,0.0885,0.0104,0.0104,0.0072,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,20,8,1.0,0.1490,0.1490,0.1490,0.0527,0.0034,0.0034,0.0021,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,25,4,1.0,0.3099,0.3099,0.3099,0.1502,0.0302,0.0302,0.0228,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,25,6,1.0,0.2502,0.2502,0.2502,0.1145,0.0177,0.0177,0.0123,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,25,8,1.0,0.1954,0.1954,0.1954,0.0841,0.0109,0.0109,0.0075,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,30,4,1.0,0.3188,0.3188,0.3188,0.1616,0.0358,0.0358,0.0279,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,30,6,1.0,0.2775,0.2775,0.2775,0.1365,0.0254,0.0254,0.0187,...,0.0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,30,8,1.0,0.2386,0.2386,0.2386,0.1100,0.0162,0.0162,0.0114,...,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
